# Baseline accuracy

Compare agent and patient next-token logits at the **last non-padding prompt token**. Accuracy means `agent_logit > patient_logit`; ties count as incorrect. This is pairwise preference, not unrestricted generation accuracy.

Use the repository's `.venv` kernel with the notebook dependencies installed (`uv sync --locked --group notebook`) and a GPU allocation for Qwen3-8B-Base in bfloat16. The checkpoint must be cached or downloadable. No activation files are needed.

Run the model-loading cell once. To try another corpus or question, edit **Experiment settings** and rerun from that cell downward. Questions must still ask for the agent; candidates are the two nouns with a leading space, each occupying one token.
Prompts are plain text completions, without a chat template.


In [1]:
import os
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

from ac_tpr.dataset import TOKENIZER_ID, TOKENIZER_REVISION, load_corpus
from ac_tpr.prompts import FEW_SHOT as DEFAULT_FEW_SHOT, TASK_TEMPLATE as DEFAULT_TASK_TEMPLATE, prepare_prompts

os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"

ROOT = Path.cwd()
if ROOT.name == "notebook":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Run from the repository root or notebook directory"


/u/dssc/zenocosini/AC-TPR-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the model once

In [2]:
MODEL_ID = TOKENIZER_ID
REVISION = TOKENIZER_REVISION
DEVICE = "cuda"
DTYPE = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION, use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=DTYPE, attn_implementation="eager",
).to(DEVICE).eval()
model.requires_grad_(False)
print({"model": MODEL_ID, "commit": model.config._commit_hash, "dtype": str(model.dtype)})


Loading weights: 100%|██████████| 399/399 [00:19<00:00, 20.16it/s]


{'model': 'Qwen/Qwen3-8B-Base', 'commit': '49e3418fbbbca6ecbdf9608b4d22e5a407081db4', 'dtype': 'torch.bfloat16'}


## Experiment settings

Keep `{sentence}` at the start of `TASK_TEMPLATE`; put demonstrations in `FEW_SHOT`. The default ends at `Answer:` with no trailing whitespace. Use the enriched corpus with its `subset` annotations. Results are stratified by active/passive voice and `names_agentive`, `nouns_contact`, and `mixed_contact`. Set `SPLITS = None` to use every split; `LIMIT` selects the first matching rows in corpus order.

In [3]:
QUESTION = "Who or what is performing the action?"
FEW_SHOT = (
    "Examples of the task:\n\n"
    f"Lucia is eating a banana. Question: {QUESTION}\nAnswer: Lucia\n\n"
    f"A man is playing a guitar. Question: {QUESTION}\nAnswer: man\n\n"
    f"The dog was loved by the girl. Question: {QUESTION}\nAnswer: girl\n\n"
    f"A ball is kicked by Alice. Question: {QUESTION}\nAnswer: Alice\n\n"
    f"An apple is touched by a rocket. Question: {QUESTION}\nAnswer: rocket\n\n"
)
TASK_TEMPLATE = "{sentence}\nQuestion: " + QUESTION + "\nAnswer:"
PROMPT_TEMPLATE = FEW_SHOT + TASK_TEMPLATE
# QUESTION = "Who or what is receiving the action?"
# FEW_SHOT = (
#     "Examples of the task:\n\n"
#     f"Lucia is eating a banana. Question: {QUESTION}\nAnswer: banana\n\n"
#     f"A man is playing a guitar. Question: {QUESTION}\nAnswer: guitar\n\n"
#     f"The dog was loved by the girl. Question: {QUESTION}\nAnswer: dog\n\n"
#     f"A ball is kicked by Alice. Question: {QUESTION}\nAnswer: ball\n\n"
#     f"An apple is touched by a rocket. Question: {QUESTION}\nAnswer: apple\n\n"
# )
# TASK_TEMPLATE = "{sentence}\nQuestion: " + QUESTION + "\nAnswer:"
# PROMPT_TEMPLATE = FEW_SHOT + TASK_TEMPLATE


CORPUS = ROOT / "data/active_pass_long/corpus.csv"
SPLITS = ["test", "gen_test"]
# FEW_SHOT = DEFAULT_FEW_SHOT  # Edit demonstrations here to try another prompt.
# TASK_TEMPLATE = DEFAULT_TASK_TEMPLATE
# PROMPT_TEMPLATE = FEW_SHOT + TASK_TEMPLATE
BATCH_SIZE = 8
LIMIT = None


In [4]:
assert BATCH_SIZE > 0
assert LIMIT is None or LIMIT > 0
corpus = load_corpus(CORPUS)
assert "subset" in corpus[0], "Subset breakdowns require the enriched active_pass_long corpus"
if SPLITS is not None:
    assert set(SPLITS) <= {row["split"] for row in corpus}, "Requested split is absent"
rows = [row for row in corpus if SPLITS is None or row["split"] in SPLITS][:LIMIT]
assert rows, "No rows selected"

prompts = [PROMPT_TEMPLATE.format(sentence=row["sentence"]) for row in rows]
input_ids, role_positions, answer_ids = prepare_prompts(
    rows, tokenizer, prompt_template=PROMPT_TEMPLATE
)
assert max(map(len, input_ids)) <= model.config.max_position_embeddings, "Prompt exceeds model context length"
print(f"{len(rows):,} examples from {CORPUS}")
display(pd.DataFrame(rows).groupby(["split", "voice", "subset"]).size().rename("n").to_frame())


2,096 examples from /orfeo/cephfs/home/dssc/zenocosini/AC-TPR-project/data/active_pass_long/corpus.csv


n
split    voice   subset             
gen_test active  mixed_contact   288
                 names_agentive  216
                 nouns_contact   336
         passive mixed_contact   288
                 names_agentive  216
                 nouns_contact   336
test     active  mixed_contact    40
                 names_agentive   98
                 nouns_contact    70
         passive mixed_contact    40
                 names_agentive   98
                 nouns_contact    70

## Inspect the prompt

In [5]:
EXAMPLE = 0
print(prompts[EXAMPLE])
print("\nExact prompt:", repr(prompts[EXAMPLE]))
print("Last prompt token:", repr(tokenizer.decode([input_ids[EXAMPLE][-1]])))
for role, token_id in zip(["agent", "patient"], answer_ids[EXAMPLE].tolist()):
    print(f"{role}: {tokenizer.decode([token_id])!r} (token ID {token_id})")


Examples of the task:

Lucia is eating a banana. Question: Who or what is performing the action?
Answer: Lucia

A man is playing a guitar. Question: Who or what is performing the action?
Answer: man

The dog was loved by the girl. Question: Who or what is performing the action?
Answer: girl

A ball is kicked by Alice. Question: Who or what is performing the action?
Answer: Alice

An apple is touched by a rocket. Question: Who or what is performing the action?
Answer: rocket

Mary kicked John.
Question: Who or what is performing the action?
Answer:

Exact prompt: 'Examples of the task:\n\nLucia is eating a banana. Question: Who or what is performing the action?\nAnswer: Lucia\n\nA man is playing a guitar. Question: Who or what is performing the action?\nAnswer: man\n\nThe dog was loved by the girl. Question: Who or what is performing the action?\nAnswer: girl\n\nA ball is kicked by Alice. Question: Who or what is performing the action?\nAnswer: Alice\n\nAn apple is touched by a rocket. 

## Generate an answer

Run after loading the model and experiment settings. Paste a dataset sentence below (or use `rows[EXAMPLE]["sentence"]`), then rerun this cell. Generation uses the same `PROMPT_TEMPLATE` as the baseline and stops at EOS or 64 new tokens.

In [13]:
SENTENCE = "The artist was loved by the baby." # rows[EXAMPLE]["sentence"]  # Or paste a dataset sentence here.
prompt = PROMPT_TEMPLATE.format(sentence=SENTENCE)
inputs = tokenizer(prompt[480:], add_special_tokens=False, return_tensors="pt").to(DEVICE)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

 The artist is performing the action.


## Score the baseline

Select the final non-padding position using the attention mask, then compare the two candidate logits. No role edits are applied.


In [8]:
batches = []
for start in range(0, len(rows), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(rows))
    inputs = tokenizer.pad(
        {"input_ids": input_ids[start:end]}, padding=True, return_tensors="pt"
    ).to(DEVICE)
    with torch.inference_mode():
        output = model(**inputs, use_cache=False)
        batch_indices = torch.arange(end - start, device=DEVICE)
        last = inputs["attention_mask"].sum(dim=1) - 1
        logits = output.logits[batch_indices, last]
        scores_batch = logits.gather(1, answer_ids[start:end].to(DEVICE))
        assert torch.isfinite(scores_batch).all(), "Nonfinite answer logits"
        batches.append(scores_batch.float().cpu())
        del output, logits, scores_batch
    print(f"Scored {end:,}/{len(rows):,}", end="\r")
print()
scores = torch.cat(batches).numpy()

results = pd.DataFrame(rows)
results["prompt"] = prompts
results["agent_logit"] = scores[:, 0]
results["patient_logit"] = scores[:, 1]
results["logit_diff"] = results["patient_logit"] - results["agent_logit"]
results["correct"] = results["agent_logit"] > results["patient_logit"]
results["patient_wins"] = results["patient_logit"] > results["agent_logit"]
results["tie"] = results["agent_logit"] == results["patient_logit"]


KeyboardInterrupt: 

## Results

Accuracy uses all selected examples, including ties. Positive logit differences favor the patient. The six voice/subset groups are shown across all selected splits, followed by a separate breakdown within each split. Each group reports its own sample count and accuracy; pooled accuracy weights sentences equally. The full per-example table remains available as `results`.

In [ ]:
display(pd.DataFrame([{
    "n": len(results),
    "accuracy": results["correct"].mean(),
    "patient_preference_rate": results["patient_wins"].mean(),
    "ties": int(results["tie"].sum()),
    "mean_logit_diff": results["logit_diff"].mean(),
}], index=["overall"]))

print("By voice and subset across selected splits:")
by_voice_subset = results.groupby(["voice", "subset"]).agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    patient_preference_rate=("patient_wins", "mean"),
    ties=("tie", "sum"),
    mean_logit_diff=("logit_diff", "mean"),
)
display(by_voice_subset.style.format({
    "accuracy": "{:.1%}", "patient_preference_rate": "{:.1%}", "mean_logit_diff": "{:.3f}",
}))

print("By split, voice, and subset:")
by_split_voice_subset = results.groupby(["split", "voice", "subset"]).agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    patient_preference_rate=("patient_wins", "mean"),
    ties=("tie", "sum"),
    mean_logit_diff=("logit_diff", "mean"),
)
display(by_split_voice_subset.style.format({
    "accuracy": "{:.1%}", "patient_preference_rate": "{:.1%}", "mean_logit_diff": "{:.3f}",
}))

print("Examples with the largest patient preference:")
display(
    results.loc[results["patient_wins"]]
    .sort_values("logit_diff", ascending=False)
    .head(20)[[
        "sentence", "split", "voice", "subset", "agent", "patient",
        "agent_logit", "patient_logit", "logit_diff",
    ]]
)


,n,accuracy,patient_preference_rate,ties,mean_logit_diff
overall,2096,0.998569,0.001431,0,-7.277672


By voice and subset across selected splits:


By split, voice, and subset:


Examples with the largest patient preference:


,sentence,split,voice,subset,agent,patient,agent_logit,patient_logit,logit_diff
967,The teacher was blocked by the door.,gen_test,passive,nouns_contact,door,teacher,38.00,38.25,0.25
1731,Mary was blocked by the door.,gen_test,passive,mixed_contact,door,Mary,38.25,38.50,0.25
1755,John was blocked by the door.,test,passive,mixed_contact,door,John,38.75,39.00,0.25
